In [ ]:
#Install all of the required packages
!pip install cartopy awscli gdown
!ONNXRUNTIME=onnxruntime-gpu pip install ai-models-panguweather

In [ ]:
# Download onnx (weight files) for Pangu-Weather
!aws s3 cp --recursive --no-sign-request s3://noaa-oar-mlwp-data/colab_resources/pw ./pw/
# Download Sample Input (will be expired in a week) : 2026020912 KIM Result from Korea Meteorological Administration
!gdown "https://drive.google.com/uc?id=1146Jozo-9pVfocCDZoDI3VgKp69uU78H"
!gdown "https://drive.google.com/uc?id=1x8r5j7wrwWrl6pvEec5EfAIt-xq5-GNV"

In [ ]:
# Check the Parameters
!ai-models panguweather --fields

In [ ]:
import os
import numpy as np
import onnx
import onnxruntime as ort
import sys
from datetime import datetime

# # Start Date
# if len(sys.argv) != 2:
#     print("Usage: inference_iterative_KIM.py YYYY-MM-DD")
#     sys.exit(1)
sdate_argv = "2026020912"

# From Start Date
start_date = datetime.strptime(sdate_argv, '%Y%m%d%H')
date_str = start_date.strftime("%Y%m%d%H")

# The directory of your input and output data
input_data_dir =f'./'
output_data_dir = f'./'
os.makedirs(output_data_dir, exist_ok=True)

# Set the behavier of onnxruntime
options = ort.SessionOptions()
options.enable_cpu_mem_arena = False
options.enable_mem_pattern   = False
options.enable_mem_reuse     = False
options.intra_op_num_threads = 1
cuda_provider_options = {'arena_extend_strategy':'kSameAsRequested',}

# Initialize onnxruntime session for Pangu-Weather Models
ort_session_24 = ort.InferenceSession('./pw/pangu_weather_24.onnx', sess_options=options,providers=[('CUDAExecutionProvider', cuda_provider_options)])
# ort_session_6 = ort.InferenceSession('./pw/pangu_weather_6.onnx', sess_options=options,providers=[('CUDAExecutionProvider', cuda_provider_options)])
# ort_session_3 = ort.InferenceSession('pangu_weather_3.onnx', sess_options=options,providers=[('CUDAExecutionProvider', cuda_provider_options)])

# Load the upper-air numpy arrays
input_prs = np.load(os.path.join(input_data_dir, 'KIM_input_prs000.npy')).astype(np.float32)
# Load the surface numpy arrays
input_srf = np.load(os.path.join(input_data_dir, 'KIM_input_sfc000.npy')).astype(np.float32)

# Run the inference session
input_prs_24, input_srf_24 = input_prs, input_srf
# input_prs_6, input_srf_6 = input_prs, input_srf
# Run the inference session
# for i in range(1):
#   file_num=3*(i+1)
#   if (i+1) % 8 == 0:
#     output_prs,   output_srf   = ort_session_24.run(None, {'input':input_prs_24, 'input_surface':input_srf_24})
#     input_prs_24, input_srf_24 = output_prs, output_srf
#     input_prs_6,  input_srf_6  = output_prs, output_srf
#     input_prs,    input_srf    = output_prs, output_srf
#   elif (i+1) % 2 == 0:
#     output_prs,  output_srf  = ort_session_6.run(None, {'input':input_6_prs, 'input_surface':input_srf_6})
#     input_prs_6, input_srf_6 = output_prs, output_srf
#     input_prs,   input_srf   = output_prs, output_srf
#   else:
#     continue # 20240118 skip 3 hours
#     output_prs, output_srf = ort_session_3.run(None, {'input':input_prs, 'input_surface':input_srf})
#     input_prs,  input_srf  = output_prs, output_srf
#
#   # Save the results with '6' added to the filenames
#   np.save(os.path.join(output_data_dir, f'KIM_PW_output_prs_{file_num:03d}'), output_prs)
#   np.save(os.path.join(output_data_dir, f'KIM_PW_output_sfc_{file_num:03d}'), output_srf)

output_prs, output_srf = ort_session_24.run(None, {'input':input_prs_24, 'input_surface':input_srf_24})
np.save(os.path.join(output_data_dir, f'KIM_PW_output_prs_024'), output_prs)
np.save(os.path.join(output_data_dir, f'KIM_PW_output_sfc_024'), output_srf)

In [ ]:
# Ploting the result
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Load the saved output files
output_prs = np.load('./KIM_PW_output_prs_024.npy')
output_sfc = np.load('./KIM_PW_output_sfc_024.npy')

print(f"Shape of pressure level output (output_prs): {output_prs.shape}")
print(f"Shape of surface level output (output_sfc): {output_sfc.shape}")

# Define coordinates based on PanguWeather grid [90, 0, -90, 360] at 0.25 degree resolution
lats = np.linspace(90, -90, 721)
lons = np.linspace(0, 360 - 0.25, 1440) # Adjust for exclusive end if necessary, based on 1440 points

# Extract 2m Temperature (2t) from surface output
# Based on 'ai-models panguweather --fields', surface variables are ['msl', '10u', '10v', '2t']
# So '2t' is the 4th variable (index 3)
# Assuming output_sfc shape is (num_surface_vars, lat, lon)
temp_2m = output_sfc[3, :, :]

# Extract u and v wind components at 850hPa from pressure level output
# Based on 'ai-models panguweather --fields', pressure level variables are ['z', 'q', 't', 'u', 'v']
# Pressure levels are [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50]
# There are 13 pressure levels.
# 'u' is variable index 3, 'v' is variable index 4.
# 850hPa is level index 2 (0-indexed).
# The output_prs shape is (variable_type, pressure_level, latitude, longitude).
u_wind_850hpa = output_prs[3, 2, :, :] * 0.1
v_wind_850hpa = output_prs[4, 2, :, :] * 0.1

# --- Plotting 2m Temperature ---
plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.coastlines()

# Plot temperature in Celsius for better readability
temp_2m_celsius = temp_2m - 273.15

# Choose a colormap, e.g., 'RdBu_r' for temperature
plt.pcolormesh(lons, lats, temp_2m_celsius, transform=ccrs.PlateCarree(), cmap='RdBu_r')
plt.colorbar(label='Temperature at 2m (Celsius)')
plt.title('PanguWeather Forecast: 2m Temperature (2T)')

plt.show()

# --- Plotting Wind Vectors at 850hPa ---
plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_global()
ax.coastlines()

# Thinning out vectors for better visualization
skip = 20 # Plot every 'skip'th arrow
plt.quiver(lons[::skip], lats[::skip], u_wind_850hpa[::skip, ::skip], v_wind_850hpa[::skip, ::skip],
           transform=ccrs.PlateCarree(), color='k', scale=200)

plt.title('PanguWeather Forecast: Wind Vectors at 850hPa (U, V)')

plt.show()
